# ASR модель

Модель: exp107, CNN + BiGRU + CTC. Декодер: beam 32 + word LM, alpha=0.6, word_bonus=2.

В свежем Google Colab вставь ссылку на датасет и выполни Run all.

In [ ]:
dataset_url = ""

## 1. Установка

Клонируем репозиторий и устанавливаем зависимости из requirements.txt в Python 3.11.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

project = Path("/content/HW1_ASR")
subprocess.run(["git", "clone", "https://github.com/ignisiris/HW1_ASR.git", str(project)], check=True)
os.chdir(project)

subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "/content/asr-env"], check=True)
python = "/content/asr-env/bin/python"
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", python, "-r", "requirements.txt"], check=True)

## 2. Ресурсы модели

Скрипт скачивает веса, языковую модель и логи.

In [ ]:
subprocess.run([python, "download_resources.py"], check=True)
model_config = "experiments/inference_exp107_lm"

## 3. Инференс и метрики

Вызываем inference.py, затем calc_metrics.py.

In [ ]:
run_dir = Path("/content/inference")
run_dir.mkdir(exist_ok=True)

def recognize(data_dir, name):
    output = run_dir / name
    subprocess.run([
        python, "inference.py", "-cn", model_config,
        f"datasets.custom.data_dir={data_dir}",
        f"inferencer.save_path={output}",
    ], check=True)
    predictions = output / "custom"
    references = data_dir / "transcriptions"
    if references.is_dir():
        subprocess.run([python, "calc_metrics.py", str(references), str(predictions)], check=True)
    print("Предсказания:", predictions)
    return predictions

## 4. Готовые примеры

Шесть записей: три dev-clean и три dev-other.

In [ ]:
sample_dir = project / "demo/sample"
recognize(sample_dir, "sample")

## 5. Датасет по ссылке

In [ ]:
input_dir = run_dir / "dataset"
subprocess.run([
    python, "prepare_demo_dataset.py", dataset_url.strip(),
    "--output", str(input_dir),
], check=True)
recognize(input_dir, "user")